# Projeto 1 | Classificação de frutas com Vision Transformers

## Problema e critérios experimentais

Este experimento compara dois Vision Transformers (ViTs) na classificação de 20 frutas e variedades do Fruits-360. Cada modelo recebe uma imagem por vez. A tarefa pode apoiar o cadastro e a conferência de produtos no varejo e na agritech, mas aqui é avaliada apenas em condições controladas.

A pergunta é quanto o pré-treinamento em um grande conjunto de imagens melhora o resultado em relação a um ViT treinado do zero neste recorte. O modelo próprio é o baseline didático; o pré-treinado é a solução principal. Escolhemos as configurações usando treino e validação. O conjunto `Test` oficial fica reservado para uma única avaliação final.

O F1 macro orienta a seleção porque dá o mesmo peso a cada uma das 20 classes. Accuracy, top-3 accuracy e F1 weighted complementam a análise.

As imagens do Fruits-360 têm fundo predominantemente uniforme e foram obtidas em condições controladas. Um bom resultado neste corpus não demonstra que o modelo funcionará em prateleiras reais nem que está pronto para produção.

## Requisitos e planejamento da execução local

| Recurso | Requisito / estimativa |
|---|---|
| RAM disponível recomendada | 16 GB |
| VRAM para a configuração padrão | 6 GB |
| Tempo estimado com dados e pesos em cache | 45–150 minutos |
| Hardware de referência | NVIDIA GeForce RTX 3050 Laptop, 6 GB de VRAM, Windows, Python 3.12 |
| Protocolo considerado | batch 16 no ViT didático; batch 1 e acumulação de 16 passos no ViT-Base |

Os valores são estimativas de planejamento, não medições de pico nem garantias de tempo. Feche aplicações que ocupem a GPU. Download inicial, armazenamento, outras cargas e early stopping alteram a duração. Instale as dependências de `requirements.txt` na pasta desta atividade e selecione o kernel local. Execute a partir dessa pasta; não há requisito de T4/Colab.


In [1]:
NOMES_CLASSES = [
    "Apple Braeburn 1", "Apple Golden 1", "Apple Granny Smith 1",
    "Apple Red Delicious 1", "Banana 1", "Banana Lady Finger 1", "Banana Red 1",
    "Pepper Green 1", "Pepper Orange 1", "Pepper Red 1", "Pepper Yellow 1",
    "Tomato 1", "Tomato 2", "Tomato 3", "Tomato 4", "Avocado 1", "Kiwi 1",
    "Lemon 1", "Pineapple 1", "Strawberry 1",
]
QUANTIDADE_CLASSES = len(NOMES_CLASSES)
CLASSE_PARA_INDICE = {nome: indice for indice, nome in enumerate(NOMES_CLASSES)}
INDICE_PARA_CLASSE = {indice: nome for nome, indice in CLASSE_PARA_INDICE.items()}

assert QUANTIDADE_CLASSES == 20 and len(set(NOMES_CLASSES)) == 20
assert set(CLASSE_PARA_INDICE) == set(NOMES_CLASSES)
assert all(INDICE_PARA_CLASSE[indice] == nome for nome, indice in CLASSE_PARA_INDICE.items())

print(f"Recorte fixado: {QUANTIDADE_CLASSES} classes únicas.")


Recorte fixado: 20 classes únicas.


## Ambiente CUDA

Importamos as dependências, fixamos a semente 42 em Python, NumPy e PyTorch e registramos o nome e a VRAM da GPU. Este notebook requer CUDA para executar os modelos e as operações com tensores. Se ela não estiver disponível, a checagem interrompe a execução com `RuntimeError`.

In [2]:
from pathlib import Path
import math, random

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEMENTE_ALEATORIA = 42
random.seed(SEMENTE_ALEATORIA)
np.random.seed(SEMENTE_ALEATORIA)
torch.manual_seed(SEMENTE_ALEATORIA)
torch.cuda.manual_seed_all(SEMENTE_ALEATORIA)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA é obrigatória para este projeto.")

DISPOSITIVO = torch.device("cuda")
PROPRIEDADES_GPU = torch.cuda.get_device_properties(0)
print(PROPRIEDADES_GPU.name, f"{PROPRIEDADES_GPU.total_memory / 1024**3:.1f} GB")


NVIDIA GeForce RTX 3050 6GB Laptop GPU 6.0 GB


## Baixar e localizar o dataset

Baixamos o Fruits-360 pelo KaggleHub e procuramos, no cache retornado, as pastas `Training` e `Test` com as 20 classes definidas no passo 1. O filtro `100x100` seleciona essa versão e evita usar o ramo com imagens em tamanho original. Se nenhuma versão atender aos critérios, a função informa o problema antes do treino. O dataset é de Horea Mureșan e Mihai Oltean e tem licença CC BY-SA 4.0.

Se a versão 107 já estiver completa no cache oficial do KaggleHub, reutilizamos esses arquivos sem consultar a API. O caminho vem da configuração do KaggleHub, sem diretório fixo da máquina. Quando não há cache, o download continua usando somente `kagglehub.dataset_download("moltean/fruits")`. As verificações de classes e partições permanecem obrigatórias.

In [3]:
def localizar_particoes(raiz: Path) -> tuple[Path, Path]:
    for diretorio_treino in sorted(raiz.rglob("Training")):
        if "100x100" not in str(diretorio_treino).lower():
            continue
        diretorio_teste = diretorio_treino.parent / "Test"
        if not diretorio_teste.is_dir():
            continue
        nomes_treino = {arquivo.name for arquivo in diretorio_treino.iterdir() if arquivo.is_dir()}
        nomes_teste  = {arquivo.name for arquivo in diretorio_teste.iterdir()  if arquivo.is_dir()}
        ausentes = set(NOMES_CLASSES) - (nomes_treino & nomes_teste)
        if not ausentes:
            return diretorio_treino, diretorio_teste
    raise FileNotFoundError("Nenhum par Training/Test contém as 20 classes.")

from kagglehub.cache import Cache
from kagglehub.handle import parse_dataset_handle

# A versão 107 foi a fonte da execução anterior; o cache evita consultar a API sem necessidade.
versao_referencia = parse_dataset_handle("moltean/fruits").with_version(107)
caminho_em_cache = Cache().load_from_cache(versao_referencia)
raiz_dataset = Path(caminho_em_cache if caminho_em_cache else kagglehub.dataset_download("moltean/fruits"))
print("Origem:", "cache oficial KaggleHub, versão 107" if caminho_em_cache else "download oficial KaggleHub")
diretorio_treino, diretorio_teste = localizar_particoes(raiz_dataset)
print(diretorio_treino)
print(diretorio_teste)


Origem: cache oficial KaggleHub, versão 107
C:\Users\lord_\.cache\kagglehub\datasets\moltean\fruits\versions\107\fruits-360_100x100\fruits-360\Training
C:\Users\lord_\.cache\kagglehub\datasets\moltean\fruits\versions\107\fruits-360_100x100\fruits-360\Test


## Validar classes e explorar os dados

Criamos um índice com caminho, identificador e nome da classe para conferir as 20 classes nos dois conjuntos. A grade abre apenas as imagens que serão exibidas. Os arquivos do Fruits-360 têm 100×100 pixels e serão redimensionados para 224×224; a interpolação não acrescenta detalhes à imagem original.

In [4]:
EXTENSOES_IMAGENS = {".jpg", ".jpeg", ".png", ".bmp"}

def catalogar_particao(diretorio_particao: Path) -> pd.DataFrame:
    registros = []
    for nome in NOMES_CLASSES:
        arquivos = [arquivo for arquivo in (diretorio_particao / nome).rglob("*") if arquivo.suffix.lower() in EXTENSOES_IMAGENS]
        registros.extend({"path": str(arquivo), "label": CLASSE_PARA_INDICE[nome], "class_name": nome} for arquivo in arquivos)
    dados = pd.DataFrame(registros)
    if dados.empty:
        raise RuntimeError(f"Nenhuma imagem encontrada em {diretorio_particao}")
    return dados

dados_treino_original = catalogar_particao(diretorio_treino)
dados_teste = catalogar_particao(diretorio_teste)
contagens_classes = pd.DataFrame({
    "train": dados_treino_original.groupby("class_name").size().reindex(NOMES_CLASSES, fill_value=0),
    "test": dados_teste.groupby("class_name").size().reindex(NOMES_CLASSES, fill_value=0),
})
display(contagens_classes)

classes_ausentes = contagens_classes.index[(contagens_classes == 0).any(axis=1)].tolist()
assert len(contagens_classes) == QUANTIDADE_CLASSES
assert not classes_ausentes, (
    f"Classes sem imagens em treino ou teste: {classes_ausentes}"
)
print(f"Confirmado: {QUANTIDADE_CLASSES}/{QUANTIDADE_CLASSES} classes têm imagens em ambos os splits.")

amostra = dados_treino_original.sample(12, random_state=SEMENTE_ALEATORIA)
figura, eixos = plt.subplots(3, 4, figsize=(12, 9))
for eixo, registro in zip(eixos.flat, amostra.itertuples()):
    with Image.open(registro.path) as imagem:
        eixo.imshow(imagem.convert("RGB"))
        eixo.set_title(f"{registro.class_name}\n{imagem.size} / {imagem.mode}")
    eixo.axis("off")
plt.tight_layout()


Confirmado: 20/20 classes têm imagens em ambos os splits.


                       train  test
class_name                        
Apple Braeburn 1         492   164
Apple Golden 1           480   160
Apple Granny Smith 1     492   164
Apple Red Delicious 1    490   166
Banana 1                 490   166
Banana Lady Finger 1     450   152
Banana Red 1             490   166
Pepper Green 1           444   148
Pepper Orange 1          702   234
Pepper Red 1             666   222
Pepper Yellow 1          666   222
Tomato 1                 738   246
Tomato 2                 672   225
Tomato 3                 738   246
Tomato 4                 479   160
Avocado 1                427   143
Kiwi 1                   466   156
Lemon 1                  492   164
Pineapple 1              490   166
Strawberry 1             492   164

<Figure size 1200x900 with 12 Axes>

## Dividir e preprocessar os dados

Reservamos 15% de `Training` para validação, com estratificação por classe e semente 42. O `Test` oficial permanece intacto. No treino, usamos `RandomResizedCrop`, `HFlip` e `Rotation(15)`; na validação e no teste, `Resize` e `Normalize(0.5)` mantêm o processamento determinístico. O `Dataset` abre cada imagem quando ela é solicitada, sem carregar o corpus inteiro na RAM.

In [5]:
dados_treino, dados_validacao = train_test_split(
    dados_treino_original, test_size=0.15, random_state=SEMENTE_ALEATORIA, stratify=dados_treino_original["label"],
)
dados_treino, dados_validacao = dados_treino.reset_index(drop=True), dados_validacao.reset_index(drop=True)
dados_teste = dados_teste.reset_index(drop=True)

MEDIA_CANAIS = DESVIO_CANAIS = [0.5, 0.5, 0.5]
transformacao_treino = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA_CANAIS, DESVIO_CANAIS),
])
transformacao_avaliacao = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA_CANAIS, DESVIO_CANAIS),
])


### Dataset e carregadores

As imagens são carregadas sob demanda. Augmentation e embaralhamento ficam restritos ao treino; validação e Test compartilham o pré-processamento de avaliação.

In [6]:
class DatasetFrutas(Dataset):
    def __init__(self, dados, transformacao, retornar_dicionario=False):
        self.dados, self.transformacao = dados, transformacao
        self.retornar_dicionario = retornar_dicionario

    def __len__(self):
        return len(self.dados)

    def __getitem__(self, indice):
        registro = self.dados.iloc[indice]
        with Image.open(registro.path) as imagem:
            pixels = self.transformacao(imagem.convert("RGB"))
        rotulo = int(registro.label)
        if self.retornar_dicionario:
            return {"pixel_values": pixels, "labels": rotulo}
        return pixels, rotulo

conjunto_treino = DatasetFrutas(dados_treino, transformacao_treino)
conjunto_validacao   = DatasetFrutas(dados_validacao,   transformacao_avaliacao)
conjunto_teste  = DatasetFrutas(dados_teste,  transformacao_avaliacao)
carregador_treino = DataLoader(conjunto_treino, batch_size=16, shuffle=True,  pin_memory=True)
carregador_validacao   = DataLoader(conjunto_validacao,   batch_size=2,  shuffle=False, pin_memory=True)
carregador_teste  = DataLoader(conjunto_teste,  batch_size=2,  shuffle=False, pin_memory=True)

print(f"train={len(dados_treino)}  val={len(dados_validacao)}  test={len(dados_teste)}")

caminhos_treino = set(dados_treino["path"])
caminhos_validacao = set(dados_validacao["path"])
caminhos_teste = set(dados_teste["path"])
assert caminhos_treino.isdisjoint(caminhos_validacao), "Há imagens repetidas entre treino e validação."
assert caminhos_treino.isdisjoint(caminhos_teste), "Há imagens repetidas entre treino e teste."
assert caminhos_validacao.isdisjoint(caminhos_teste), "Há imagens repetidas entre validação e teste."
print("Confirmado: não há sobreposição entre treino, validação e teste.")

distribuicao_classes = pd.DataFrame({
    "training_original": dados_treino_original["label"].value_counts(normalize=True).reindex(
        range(QUANTIDADE_CLASSES), fill_value=0
    ),
    "validation": dados_validacao["label"].value_counts(normalize=True).reindex(
        range(QUANTIDADE_CLASSES), fill_value=0
    ),
})
distribuicao_classes.index = [INDICE_PARA_CLASSE[indice_classe] for indice_classe in distribuicao_classes.index]


train=9227  val=1629  test=3634
Confirmado: não há sobreposição entre treino, validação e teste.


### Conferir a divisão

Conferimos as proporções das classes e o formato RGB de um lote de 224 × 224. O Test oficial segue separado do desenvolvimento.

In [7]:
distribuicao_classes["difference"] = (
    distribuicao_classes["validation"] - distribuicao_classes["training_original"]
).abs()
display(distribuicao_classes)
maior_diferenca = distribuicao_classes["difference"].max()
assert maior_diferenca < 0.01, (
    f"Distribuição da validação divergiu mais de 1 ponto percentual: {maior_diferenca:.4f}"
)
print(f"Proporção preservada; maior diferença: {maior_diferenca:.4f}.")

pixels, rotulos = next(iter(carregador_treino))
assert pixels.shape[1:] == (3, 224, 224)
assert rotulos.min().item() >= 0
assert rotulos.max().item() < QUANTIDADE_CLASSES
pixels = pixels.to(DISPOSITIVO, non_blocking=True)
rotulos = rotulos.to(DISPOSITIVO, non_blocking=True)
assert pixels.is_cuda and rotulos.is_cuda
print(
    "Lote transferido para CUDA:", tuple(pixels.shape),
    f"labels={rotulos.min().item()}..{rotulos.max().item()}",
    f"device={pixels.device}",
)


Proporção preservada; maior diferença: 0.0003.
Lote transferido para CUDA: (16, 3, 224, 224) labels=0..19 device=cuda:0


                       training_original  validation  difference
Apple Braeburn 1                0.045321    0.045427    0.000106
Apple Golden 1                  0.044215    0.044199    0.000016
Apple Granny Smith 1            0.045321    0.045427    0.000106
Apple Red Delicious 1           0.045136    0.044813    0.000324
Banana 1                        0.045136    0.044813    0.000324
Banana Lady Finger 1            0.041452    0.041130    0.000322
Banana Red 1                    0.045136    0.045427    0.000290
Pepper Green 1                  0.040899    0.041130    0.000230
Pepper Orange 1                 0.064665    0.064457    0.000208
Pepper Red 1                    0.061349    0.061387    0.000039
Pepper Yellow 1                 0.061349    0.061387    0.000039
Tomato 1                        0.067981    0.068140    0.000159
Tomato 2                        0.061901    0.062001    0.000100
Tomato 3                        0.067981    0.068140    0.000159
Tomato 4                 

## Scaled dot-product attention

A atenção recebe consultas ($Q$), chaves ($K$) e valores ($V$), separados por head. O produto $QK^T$ compara cada consulta com todas as chaves. Dividimos os scores por $\sqrt{d_{head}}$ para evitar que cresçam com a dimensão do head e saturem o softmax. Aplicado ao último eixo, o softmax produz os pesos que combinam os valores.

Na classificação de imagens, todos os patches podem se relacionar, então não usamos máscara causal. Retornamos os pesos antes do dropout para preservar sua soma normalizada e permitir a inspeção posterior.

In [8]:
class AtencaoProdutoEscalar(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, consultas, chaves, valores):
        escala = consultas.size(-1) ** -0.5
        pontuacoes = (consultas @ chaves.transpose(-2, -1)) * escala
        pesos = pontuacoes.softmax(dim=-1)
        contexto = self.dropout(pesos) @ valores
        return contexto, pesos

atencao = AtencaoProdutoEscalar().to(DISPOSITIVO)
consultas_verificacao = chaves_verificacao = valores_verificacao = torch.randn(2, 6, 197, 32, device=DISPOSITIVO)
contexto, pesos = atencao(consultas_verificacao, chaves_verificacao, valores_verificacao)

assert contexto.shape == (2, 6, 197, 32)
assert pesos.shape == (2, 6, 197, 197)
somas_linhas = pesos.sum(dim=-1)
assert torch.allclose(somas_linhas, torch.ones_like(somas_linhas), atol=1e-5)
pesos_esperados = ((consultas_verificacao @ chaves_verificacao.transpose(-2, -1)) / math.sqrt(consultas_verificacao.size(-1))).softmax(dim=-1)
assert torch.allclose(pesos, pesos_esperados)
assert contexto.is_cuda and pesos.is_cuda

print(f"Contexto: {tuple(contexto.shape)} em {contexto.device}")
print(f"Pesos: {tuple(pesos.shape)} em {pesos.device}")
print(
    "Soma por linha dos pesos:",
    f"mínima={somas_linhas.min().item():.6f}",
    f"máxima={somas_linhas.max().item():.6f}",
)


Contexto: (2, 6, 197, 32) em cuda:0
Pesos: (2, 6, 197, 197) em cuda:0
Soma por linha dos pesos: mínima=1.000000 máxima=1.000000


## Projeções independentes por head

Cada head tem projeções lineares próprias para consultas, chaves e valores. Concatenamos os seis contextos de 32 dimensões para recuperar a dimensão 192 e aplicamos a projeção de saída. A divisão pela raiz da dimensão das chaves controla a magnitude dos scores.

In [9]:
class CabecaAtencao(nn.Module):
    """Cada head aprende suas próprias projeções de consulta, chave e valor."""
    def __init__(self, dimensao_embedding, dimensao_head, dropout):
        super().__init__()
        self.projetar_consultas = nn.Linear(dimensao_embedding, dimensao_head)
        self.projetar_chaves = nn.Linear(dimensao_embedding, dimensao_head)
        self.projetar_valores = nn.Linear(dimensao_embedding, dimensao_head)
        self.atencao = AtencaoProdutoEscalar(dropout)

    def forward(self, tokens):
        consultas = self.projetar_consultas(tokens)
        chaves = self.projetar_chaves(tokens)
        valores = self.projetar_valores(tokens)
        return self.atencao(consultas, chaves, valores)


### Combinar os heads

Cada head aprende relações entre os tokens. Reunimos seus contextos por concatenação e aplicamos a projeção de saída.

In [10]:
class AtencaoMultiHead(nn.Module):
    def __init__(self, dimensao_embedding=192, quantidade_heads=6, dropout=0.1):
        super().__init__()
        if dimensao_embedding % quantidade_heads != 0:
            raise ValueError("A dimensão precisa ser divisível pela quantidade de heads.")
        self.dimensao_head = dimensao_embedding // quantidade_heads
        self.cabecas = nn.ModuleList([
            CabecaAtencao(dimensao_embedding, self.dimensao_head, dropout) for _ in range(quantidade_heads)
        ])
        self.projecao_saida = nn.Linear(dimensao_embedding, dimensao_embedding)

    def forward(self, tokens):
        contextos, atencoes = [], []
        for cabeca in self.cabecas:
            contexto, pesos = cabeca(tokens)
            contextos.append(contexto)
            atencoes.append(pesos)
        contextos_concatenados = torch.cat(contextos, dim=-1)
        return self.projecao_saida(contextos_concatenados), torch.stack(atencoes, dim=1)


### Encoder com residuais

O encoder aplica LayerNorm, attention e um feedforward de duas camadas. As duas conexões residuais mantêm um caminho para a representação de entrada.

In [11]:
class BlocoTransformer(nn.Module):
    def __init__(self, dimensao_embedding=192, quantidade_heads=6, fator_expansao=4, dropout=0.1):
        super().__init__()
        self.normalizacao_atencao = nn.LayerNorm(dimensao_embedding)
        self.normalizacao_feedforward = nn.LayerNorm(dimensao_embedding)
        self.atencao = AtencaoMultiHead(dimensao_embedding, quantidade_heads, dropout)
        self.feedforward = nn.Sequential(
            nn.Linear(dimensao_embedding, dimensao_embedding * fator_expansao), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dimensao_embedding * fator_expansao, dimensao_embedding), nn.Dropout(dropout),
        )

    def forward(self, tokens):
        tokens_atendidos, pesos = self.atencao(self.normalizacao_atencao(tokens))
        tokens = tokens + tokens_atendidos
        tokens = tokens + self.feedforward(self.normalizacao_feedforward(tokens))
        return tokens, pesos


bloco = BlocoTransformer().to(DISPOSITIVO)
tokens_entrada = torch.randn(2, 197, 192, device=DISPOSITIVO)
tokens_codificados, pesos_encoder = bloco(tokens_entrada)
assert tokens_codificados.shape == tokens_entrada.shape
assert pesos_encoder.shape == (2, 6, 197, 197)
assert torch.allclose(pesos_encoder.sum(-1), torch.ones_like(pesos_encoder.sum(-1)), atol=1e-5)
tokens_codificados.square().mean().backward()
for cabeca in bloco.atencao.cabecas:
    for parametro in cabeca.parameters():
        assert parametro.grad is not None and torch.isfinite(parametro.grad).all()
for parametro in bloco.feedforward.parameters():
    assert parametro.grad is not None and torch.isfinite(parametro.grad).all()
assert len({id(cabeca.projetar_consultas.weight) for cabeca in bloco.atencao.cabecas}) == 6
print("Heads independentes, concatenação, residuais e gradientes verificados.")
bloco.zero_grad(set_to_none=True)
tokens_codificados = tokens_codificados.detach()
pesos_encoder = pesos_encoder.detach()


Heads independentes, concatenação, residuais e gradientes verificados.


## Vision Transformer completo

Transformamos cada patch de 16 × 16 em um token. Um token CLS aprendível reúne a informação usada na classificação, e o positional encoding informa a posição dos patches. Sem as posições, self-attention relaciona conteúdos, mas não reconhece onde eles estavam na grade: permutar patches permuta suas saídas e retira do resumo CLS a referência espacial.

O modelo usa 196 patches, dimensão 192, seis blocos e seis heads. Os logits correspondem às classes. Em modo de avaliação, solicitar as atenções não altera esses logits.

In [12]:
class PatchEmbedding(nn.Module):
    def __init__(self, tamanho_patch, dimensao_embedding):
        super().__init__()
        self.projecao = nn.Conv2d(3, dimensao_embedding, tamanho_patch, stride=tamanho_patch)

    def forward(self, imagens):
        return self.projecao(imagens).flatten(2).transpose(1, 2)


### Compor o modelo

Os encoders processam o token CLS e os patches com suas posições. A cabeça classificadora usa o resumo CLS para produzir os logits.

In [13]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        tamanho_imagem=224,
        tamanho_patch=16,
        dimensao_embedding=192,
        quantidade_blocos=6,
        quantidade_heads=6,
        fator_expansao=4,
        dropout=0.1,
        quantidade_classes=QUANTIDADE_CLASSES,
    ):
        super().__init__()
        if tamanho_imagem % tamanho_patch != 0:
            raise ValueError("image_size deve ser divisível por patch_size.")
        self.tamanho_imagem = tamanho_imagem
        self.quantidade_patches = (tamanho_imagem // tamanho_patch) ** 2
        self.patch_embedding = PatchEmbedding(tamanho_patch, dimensao_embedding)
        self.token_classificacao = nn.Parameter(torch.zeros(1, 1, dimensao_embedding))
        self.posicoes = nn.Parameter(torch.zeros(1, self.quantidade_patches + 1, dimensao_embedding))
        self.dropout = nn.Dropout(dropout)
        self.blocos = nn.ModuleList([
            BlocoTransformer(dimensao_embedding, quantidade_heads, fator_expansao, dropout) for _ in range(quantidade_blocos)
        ])
        self.normalizacao_final = nn.LayerNorm(dimensao_embedding)
        self.classificador = nn.Linear(dimensao_embedding, quantidade_classes)
        nn.init.trunc_normal_(self.token_classificacao, std=0.02)
        nn.init.trunc_normal_(self.posicoes, std=0.02)

    def forward(self, imagens, retornar_atencoes=False):
        if imagens.shape[-2:] != (self.tamanho_imagem, self.tamanho_imagem):
            raise ValueError(
                f"Esperado {self.tamanho_imagem}×{self.tamanho_imagem}, "
                f"recebido {imagens.shape[-2]}×{imagens.shape[-1]}."
            )
        tokens = self.patch_embedding(imagens)
        token_classificacao = self.token_classificacao.expand(imagens.size(0), -1, -1)
        tokens = torch.cat([token_classificacao, tokens], dim=1)
        tokens = self.dropout(tokens + self.posicoes)

        atencoes = []
        for bloco in self.blocos:
            tokens, pesos = bloco(tokens)
            if retornar_atencoes:
                atencoes.append(pesos)

        resumo_classificacao = self.normalizacao_final(tokens)[:, 0]
        logits = self.classificador(resumo_classificacao)
        return (logits, atencoes) if retornar_atencoes else logits


### Verificar o ViT completo

Conferimos o formato dos patches, do CLS, das posições e dos logits. Também verificamos se retornar attention weights preserva os logits em avaliação.

In [14]:
vit_didatico = VisionTransformer().to(DISPOSITIVO).eval()
imagens = torch.randn(2, 3, 224, 224, device=DISPOSITIVO)
with torch.inference_mode():
    tokens_patches = vit_didatico.patch_embedding(imagens)
    token_classificacao_expandido = vit_didatico.token_classificacao.expand(imagens.size(0), -1, -1)
    sequencia_inicial = torch.cat([token_classificacao_expandido, tokens_patches], dim=1)
    logits, atencoes = vit_didatico(imagens, retornar_atencoes=True)
    logits_sem_atencoes = vit_didatico(imagens)

assert tokens_patches.shape == (2, 196, 192)
assert sequencia_inicial.shape == (2, 197, 192)
assert torch.equal(sequencia_inicial[:, :1], token_classificacao_expandido)
assert vit_didatico.posicoes.shape == (1, 197, 192)
assert logits.shape == (2, QUANTIDADE_CLASSES)
assert len(atencoes) == 6
assert all(pesos.shape == (2, 6, 197, 197) for pesos in atencoes)
assert torch.allclose(logits, logits_sem_atencoes)
assert logits.is_cuda and all(pesos.is_cuda for pesos in atencoes)

quantidade_parametros = sum(parametro.numel() for parametro in vit_didatico.parameters())
print(f"Patch tokens: {tuple(tokens_patches.shape)}")
print(f"Sequência com [CLS]: {tuple(sequencia_inicial.shape)}")
print(f"Logits: {tuple(logits.shape)}")
print(f"Atenções por bloco: {len(atencoes)} × {tuple(atencoes[-1].shape)}")
print(f"Parâmetros treináveis: {quantidade_parametros:,}")


Patch tokens: (2, 196, 192)
Sequência com [CLS]: (2, 197, 192)
Logits: (2, 20)
Atenções por bloco: 6 × (2, 6, 197, 197)
Parâmetros treináveis: 2,859,092


## Treinar o ViT do zero

Treinamos o baseline apenas com `carregador_treino`. A cada época, o F1 macro de `carregador_validacao` determina qual checkpoint será guardado. O `Test` oficial fica fora do treino, do early stopping e da seleção. AdamW usa learning rate de 3e-4 e weight decay de 0,05; a entropia cruzada usa label smoothing de 0,1. O treino combina FP16, `GradScaler`, clipping de gradiente em 1,0 e scheduler cosseno. Ele termina após três épocas consecutivas sem melhora.

Salvamos somente o `state_dict` e recarregamos o melhor estado ao final. A tabela e a curva usam os valores registrados nesta execução.

In [15]:
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

CAMINHO_RESULTADOS = Path("resultados/v3")
CAMINHO_RESULTADOS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_VIT_DIDATICO = CAMINHO_RESULTADOS / "custom_vit_best.pt"

assert carregador_treino.batch_size == 16

@torch.inference_mode()
def calcular_f1_validacao(modelo, carregador):
    modelo.eval()
    rotulos_reais, rotulos_previstos = [], []
    for imagens, rotulos in carregador:
        imagens = imagens.to(DISPOSITIVO, non_blocking=True)
        logits = modelo(imagens)
        assert imagens.is_cuda and logits.is_cuda
        rotulos_reais.extend(rotulos.numpy())
        rotulos_previstos.extend(logits.argmax(dim=1).cpu().numpy())
    return f1_score(
        rotulos_reais, rotulos_previstos, average="macro", zero_division=0
    )


### Treinar o modelo didático

Executamos treino e validação com early stopping. Ao terminar, restauramos o checkpoint escolhido pela validação.

In [16]:
def executar_epoca_vit(modelo, otimizador, escalador):
    modelo.train()
    perda_total, quantidade_exemplos = 0.0, 0
    for imagens, rotulos in tqdm(carregador_treino, desc="Treino", leave=False):
        imagens = imagens.to(DISPOSITIVO, non_blocking=True)
        rotulos = rotulos.to(DISPOSITIVO, non_blocking=True)
        otimizador.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = modelo(imagens)
            perda = F.cross_entropy(logits, rotulos, label_smoothing=0.1)
        escalador.scale(perda).backward()
        escalador.unscale_(otimizador)
        nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
        escalador.step(otimizador)
        escalador.update()
        perda_total += perda.item() * rotulos.size(0)
        quantidade_exemplos += rotulos.size(0)
    return perda_total / quantidade_exemplos


def treinar_vit_didatico(modelo, epocas=15):
    otimizador = torch.optim.AdamW(modelo.parameters(), lr=3e-4, weight_decay=0.05)
    agendador = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, T_max=epocas)
    escalador = torch.amp.GradScaler("cuda")
    melhor_f1, epocas_sem_melhora, historico = -1.0, 0, []
    for epoca in range(1, epocas + 1):
        perda_treino = executar_epoca_vit(modelo, otimizador, escalador)
        pontuacao = calcular_f1_validacao(modelo, carregador_validacao)
        historico.append({"epoch": epoca, "train_loss": perda_treino,
                          "val_f1_macro": pontuacao, "learning_rate": otimizador.param_groups[0]["lr"]})
        if pontuacao > melhor_f1:
            melhor_f1, epocas_sem_melhora = pontuacao, 0
            torch.save(modelo.state_dict(), CHECKPOINT_VIT_DIDATICO)
        else:
            epocas_sem_melhora += 1
        print(f"Época {epoca:02d}: loss={perda_treino:.4f}, F1 validação={pontuacao:.4f}")
        agendador.step()
        if epocas_sem_melhora >= 3:
            print("Early stopping: três épocas sem melhora.")
            break
    modelo.load_state_dict(torch.load(CHECKPOINT_VIT_DIDATICO, map_location=DISPOSITIVO, weights_only=True))
    modelo.eval()
    return pd.DataFrame(historico)


### Executar o treino do zero

Liberamos os tensores usados nos testes e iniciamos o treino com o orçamento previsto. Depois, recarregamos o estado com maior F1 macro de validação.

In [17]:
del imagens, logits, logits_sem_atencoes, atencoes
torch.cuda.empty_cache()
historico_vit_didatico = treinar_vit_didatico(vit_didatico)
display(historico_vit_didatico)
print(f"Melhor checkpoint recarregado: {CHECKPOINT_VIT_DIDATICO.resolve()}")
print(f"Melhor F1 macro de validação: {historico_vit_didatico['val_f1_macro'].max():.4f}")


Época 01: loss=1.0982, F1 validação=0.9263
Época 02: loss=0.7192, F1 validação=0.8908
Época 03: loss=0.6555, F1 validação=0.9918
Época 04: loss=0.6401, F1 validação=0.9896
Época 05: loss=0.6266, F1 validação=0.9951
Época 06: loss=0.6142, F1 validação=0.9440
Época 07: loss=0.6032, F1 validação=0.9861
Época 08: loss=0.6065, F1 validação=0.8975
Early stopping: três épocas sem melhora.
Melhor checkpoint recarregado: C:\workspace\python\projeto-3-modulo-2\projeto-1\resultados\v3\custom_vit_best.pt
Melhor F1 macro de validação: 0.9951


   epoch  train_loss  val_f1_macro  learning_rate
0      1    1.098171      0.926255       0.000300
1      2    0.719192      0.890796       0.000297
2      3    0.655519      0.991846       0.000287
3      4    0.640144      0.989624       0.000271
4      5    0.626555      0.995077       0.000250
5      6    0.614229      0.944025       0.000225
6      7    0.603213      0.986106       0.000196
7      8    0.606459      0.897505       0.000166

### Curvas do modelo didático

A loss acompanha o ajuste aos dados de treino. O F1 macro de validação determina o estado escolhido.

In [18]:
figura, eixo_perda = plt.subplots(figsize=(9, 5))
eixo_f1 = eixo_perda.twinx()
eixo_perda.plot(
    historico_vit_didatico["epoch"], historico_vit_didatico["train_loss"],
    marker="o", color="tab:blue", label="Perda de treino",
)
eixo_f1.plot(
    historico_vit_didatico["epoch"], historico_vit_didatico["val_f1_macro"],
    marker="s", color="tab:orange", label="F1 macro de validação",
)
eixo_perda.set(
    xlabel="Época", ylabel="Perda de treino",
    title="Treino do ViT do zero",
)
eixo_f1.set_ylabel("F1 macro de validação")
eixo_perda.grid(alpha=0.25)
linhas = eixo_perda.get_lines() + eixo_f1.get_lines()
eixo_perda.legend(linhas, [linha.get_label() for linha in linhas], loc="center right")
plt.tight_layout()


<Figure size 900x500 with 2 Axes>

## Fine-tuning do ViT pré-treinado

Carregamos o ViT-Base pré-treinado no ImageNet-21k e adaptamos a cabeça às 20 classes. Treinamos apenas essa cabeça por até duas épocas; depois, liberamos o backbone por até cinco épocas, com learning rate menor. O F1 macro de validação continua orientando a seleção. Para caber na GPU local, usamos batch 1, acumulação de 16 passos, mixed precision e gradient checkpointing.

A restauração do melhor estado usa `from_pretrained` para converter nomes antigos de pesos e depois `load_state_dict(strict=True)`. Qualquer chave ausente, inesperada ou incompatível interrompe o pipeline. Conferimos a igualdade de todos os tensores; a cabeça e o backbone precisam corresponder ao checkpoint escolhido pela validação.

Depois de cada restauração, recalculamos o F1 macro na validação e exigimos igualdade com a métrica registrada para o melhor checkpoint. Essa conferência não usa o Test nem altera hiperparâmetros.

No carregamento inicial do ImageNet-21k, a nova cabeça aparece como não carregada e o pooler original pode ser informado como não utilizado; isso decorre da adaptação à classificação de 20 classes. Nas restaurações dos checkpoints desta tarefa, todas as chaves devem corresponder e a auditoria exige zero problemas.

In [19]:
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    DefaultDataCollator, EarlyStoppingCallback, Trainer, TrainingArguments,
)

CHECKPOINT_PRE_TREINADO = "google/vit-base-patch16-224-in21k"
CAMINHO_TREINOS_PRE_TREINADO = CAMINHO_RESULTADOS / "hf_outputs"
CAMINHO_MODELO_FINAL = CAMINHO_RESULTADOS / "vit_fruits360_best"

processador = AutoImageProcessor.from_pretrained(CHECKPOINT_PRE_TREINADO)
tamanho_processador = processador.size
assert tamanho_processador["height"] == tamanho_processador["width"] == 224
assert np.allclose(processador.image_mean, [0.5] * 3)
assert np.allclose(processador.image_std, [0.5] * 3)

conjunto_treino_pre_treinado = DatasetFrutas(dados_treino, transformacao_treino, retornar_dicionario=True)
conjunto_validacao_pre_treinado = DatasetFrutas(dados_validacao, transformacao_avaliacao, retornar_dicionario=True)
conjunto_teste_pre_treinado = DatasetFrutas(dados_teste, transformacao_avaliacao, retornar_dicionario=True)

vit_pre_treinado = AutoModelForImageClassification.from_pretrained(
    CHECKPOINT_PRE_TREINADO,
    num_labels=QUANTIDADE_CLASSES,
    id2label=INDICE_PARA_CLASSE,
    label2id=CLASSE_PARA_INDICE,
    ignore_mismatched_sizes=True,
    attn_implementation="eager",
).to(DISPOSITIVO)
vit_pre_treinado.gradient_checkpointing_enable()


class TreinadorComRestauracaoEstrita(Trainer):
    """Converte o formato salvo com a API oficial e exige todos os pesos."""
    def _load_best_model(self):
        caminho = self.state.best_model_checkpoint
        if caminho is None:
            raise RuntimeError("O treinamento não registrou um melhor checkpoint.")
        restaurado, informacoes = AutoModelForImageClassification.from_pretrained(
            caminho, output_loading_info=True, attn_implementation="eager",
            local_files_only=True,
        )
        problemas = {nome: list(informacoes.get(nome, [])) for nome in
                     ("missing_keys", "unexpected_keys", "mismatched_keys", "error_msgs")}
        if any(problemas.values()):
            raise RuntimeError(f"Checkpoint incompleto ou incompatível: {problemas}")
        # from_pretrained aplica a conversão oficial de nomes do Transformers.
        # load_state_dict do PyTorch, isoladamente, não aplica essa conversão.
        estado = restaurado.state_dict()
        self.model.load_state_dict(estado, strict=True)
        for nome, valor in self.model.state_dict().items():
            if not torch.equal(valor.detach().cpu(), estado[nome].detach().cpu()):
                raise AssertionError(f"Peso não restaurado integralmente: {nome}")
        self.auditoria_restauracao = {
            "checkpoint": str(Path(caminho).resolve()),
            "quantidade_tensores": len(estado), "strict": True,
            "problemas": problemas, "igualdade_de_todos_os_tensores": True,
        }
        print("Restauração estrita aprovada:", self.auditoria_restauracao)
        del restaurado, estado


Loading weights: 100%|##########| 198/198 [00:00<00:00, 3938.69it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Seleção pelo Trainer

O Trainer calcula o F1 macro de validação e segue os limites de épocas, learning rates e early stopping definidos no protocolo.

In [20]:
def calcular_metricas_treinador(resultado_validacao):
    logits, rotulos = resultado_validacao
    previsoes = np.asarray(logits).argmax(axis=-1)
    return {
        "f1_macro": f1_score(
            rotulos, previsoes, average="macro", zero_division=0
        )
    }


def criar_argumentos_treino(pasta, epocas, taxa_aprendizado):
    return TrainingArguments(
        output_dir=str(pasta),
        num_train_epochs=epocas,
        learning_rate=taxa_aprendizado,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=16,
        fp16=True,
        gradient_checkpointing=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        report_to="none",
        remove_unused_columns=False,
        seed=SEMENTE_ALEATORIA,
        data_seed=SEMENTE_ALEATORIA,
    )


### Montar o Trainer

Configuramos o Trainer com o modelo, os datasets e as métricas. O early stopping acompanha a validação.

In [21]:
def criar_treinador(argumentos):
    return TreinadorComRestauracaoEstrita(
        model=vit_pre_treinado,
        args=argumentos,
        train_dataset=conjunto_treino_pre_treinado,
        eval_dataset=conjunto_validacao_pre_treinado,
        data_collator=DefaultDataCollator(),
        compute_metrics=calcular_metricas_treinador,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )


with torch.inference_mode():
    saida_verificacao = vit_pre_treinado(
        pixel_values=conjunto_validacao_pre_treinado[0]["pixel_values"].unsqueeze(0).to(DISPOSITIVO),
        output_attentions=True,
    )
assert saida_verificacao.logits.shape == (1, QUANTIDADE_CLASSES)
assert saida_verificacao.attentions[-1].shape == (1, 12, 197, 197)
assert saida_verificacao.logits.is_cuda
del saida_verificacao
torch.cuda.empty_cache()
print("Processador:", tamanho_processador, processador.image_mean, processador.image_std)
print("Forward ViT-Base aprovado: logits [1, 20], atenção [1, 12, 197, 197].")


Processador: SizeDict(height=224, width=224, longest_edge=None, shortest_edge=None, max_height=None, max_width=None, min_pixels=None, max_pixels=None) (0.5, 0.5, 0.5) (0.5, 0.5, 0.5)
Forward ViT-Base aprovado: logits [1, 20], atenção [1, 12, 197, 197].


### Treinar a cabeça

Com o backbone congelado, ajustamos a nova cabeça às classes de frutas.

In [22]:
for parametro in vit_pre_treinado.vit.parameters():
    parametro.requires_grad = False
for parametro in vit_pre_treinado.classifier.parameters():
    parametro.requires_grad = True

parametros_cabeca_treinaveis = [
    nome for nome, parametro in vit_pre_treinado.named_parameters()
    if parametro.requires_grad
]
assert parametros_cabeca_treinaveis == ["classifier.weight", "classifier.bias"]
print("Fase 1 — parâmetros treináveis:", parametros_cabeca_treinaveis)

treinador_cabeca = criar_treinador(
    criar_argumentos_treino(CAMINHO_TREINOS_PRE_TREINADO / "head", 2, 1e-3)
)
resultado_treino_cabeca = treinador_cabeca.train()
metricas_estado_restaurado = treinador_cabeca.evaluate()
assert np.isclose(metricas_estado_restaurado['eval_f1_macro'], treinador_cabeca.state.best_metric, atol=1e-8)
treinador_cabeca.auditoria_restauracao['f1_macro_validacao_recalculado'] = metricas_estado_restaurado['eval_f1_macro']
treinador_cabeca.auditoria_restauracao['f1_macro_validacao_selecionado'] = treinador_cabeca.state.best_metric
print('F1 da validação após restauração:', metricas_estado_restaurado['eval_f1_macro'])

historico_cabeca = pd.DataFrame(treinador_cabeca.state.log_history)
display(historico_cabeca.dropna(how="all", subset=["loss", "eval_f1_macro"]))


Fase 1 — parâmetros treináveis: ['classifier.weight', 'classifier.bias']
{'loss': '0.5175', 'grad_norm': '0.2951', 'learning_rate': '0.0005009', 'epoch': '1'}
{'eval_loss': '0.1724', 'eval_f1_macro': '0.9787', 'eval_runtime': '27.73', 'eval_samples_per_second': '58.75', 'eval_steps_per_second': '29.39', 'epoch': '1'}
{'loss': '0.1295', 'grad_norm': '0.5422', 'learning_rate': '8.666e-07', 'epoch': '2'}
{'eval_loss': '0.1261', 'eval_f1_macro': '0.9821', 'eval_runtime': '27.29', 'eval_samples_per_second': '59.69', 'eval_steps_per_second': '29.86', 'epoch': '2'}
{'train_runtime': '564.3', 'train_samples_per_second': '32.7', 'train_steps_per_second': '2.045', 'train_loss': '0.3235', 'epoch': '2'}
Restauração estrita aprovada: {'checkpoint': 'C:\\workspace\\python\\projeto-3-modulo-2\\projeto-1\\resultados\\v3\\hf_outputs\\head\\checkpoint-1154', 'quantidade_tensores': 200, 'strict': True, 'problemas': {'missing_keys': [], 'unexpected_keys': [], 'mismatched_keys': [], 'error_msgs': []}, 'igu

       loss  grad_norm  ...  total_flos  train_loss
0  0.517518   0.295133  ...         NaN         NaN
1       NaN        NaN  ...         NaN         NaN
2  0.129496   0.542156  ...         NaN         NaN
3       NaN        NaN  ...         NaN         NaN
5       NaN        NaN  ...         NaN         NaN

[5 rows x 15 columns]

 99%|#########9| 810/815 [00:27<00:00, 28.16it/s]
                                                  
100%|##########| 815/815 [00:27<00:00, 27.86it/s]
                                                 
 99%|#########9| 810/815 [00:27<00:00, 28.79it/s]
                                                   
100%|##########| 815/815 [00:27<00:00, 28.64it/s]
                                                 
100%|##########| 815/815 [00:29<00:00, 27.54it/s]


### Fine-tuning do backbone

Descongelamos o backbone e usamos learning rate menor. Escolhemos o estado pela validação, mantendo o Test reservado.

In [23]:
for parametro in vit_pre_treinado.vit.parameters():
    parametro.requires_grad = True

backbone_treinavel = all(
    parametro.requires_grad for parametro in vit_pre_treinado.vit.parameters()
)
assert backbone_treinavel
print("Fase 2 — backbone totalmente descongelado:", backbone_treinavel)

treinador_fine_tuning = criar_treinador(
    criar_argumentos_treino(CAMINHO_TREINOS_PRE_TREINADO / "full", 5, 2e-5)
)
resultado_fine_tuning = treinador_fine_tuning.train()
metricas_estado_restaurado = treinador_fine_tuning.evaluate()
assert np.isclose(metricas_estado_restaurado['eval_f1_macro'], treinador_fine_tuning.state.best_metric, atol=1e-8)
treinador_fine_tuning.auditoria_restauracao['f1_macro_validacao_recalculado'] = metricas_estado_restaurado['eval_f1_macro']
treinador_fine_tuning.auditoria_restauracao['f1_macro_validacao_selecionado'] = treinador_fine_tuning.state.best_metric
print('F1 da validação após restauração:', metricas_estado_restaurado['eval_f1_macro'])

treinador_fine_tuning.save_model(str(CAMINHO_MODELO_FINAL))
processador.save_pretrained(str(CAMINHO_MODELO_FINAL))
historico_fine_tuning = pd.DataFrame(treinador_fine_tuning.state.log_history)
display(historico_fine_tuning.dropna(how="all", subset=["loss", "eval_f1_macro"]))
print("Modelo e processador finais:", CAMINHO_MODELO_FINAL.resolve())

figura, eixos = plt.subplots(1, 2, figsize=(12, 4))
for rotulo, historico, cor in [
    ("Cabeça", historico_cabeca, "tab:blue"),
    ("Fine-tuning", historico_fine_tuning, "tab:orange"),
]:
    registros_treino = historico.dropna(subset=["loss"])
    registros_validacao = historico.dropna(subset=["eval_f1_macro"])
    eixos[0].plot(registros_treino["epoch"], registros_treino["loss"], marker="o", label=rotulo, color=cor)
    eixos[1].plot(registros_validacao["epoch"], registros_validacao["eval_f1_macro"], marker="o", label=rotulo, color=cor)
eixos[0].set(title="Perda de treino", xlabel="Época", ylabel="Loss")
eixos[1].set(title="Seleção na validação", xlabel="Época", ylabel="F1 macro")
for eixo in eixos:
    eixo.grid(alpha=0.25)
    eixo.legend()
plt.tight_layout()


Fase 2 — backbone totalmente descongelado: True
{'loss': '0.007548', 'grad_norm': '0.01348', 'learning_rate': '1.601e-05', 'epoch': '1'}
{'eval_loss': '0.003442', 'eval_f1_macro': '1', 'eval_runtime': '27.91', 'eval_samples_per_second': '58.36', 'eval_steps_per_second': '29.2', 'epoch': '1'}
{'loss': '0.0006622', 'grad_norm': '0.005291', 'learning_rate': '1.201e-05', 'epoch': '2'}
{'eval_loss': '0.0005634', 'eval_f1_macro': '1', 'eval_runtime': '27.34', 'eval_samples_per_second': '59.58', 'eval_steps_per_second': '29.81', 'epoch': '2'}
{'loss': '0.001042', 'grad_norm': '0.001571', 'learning_rate': '8.007e-06', 'epoch': '3'}
{'eval_loss': '0.0003714', 'eval_f1_macro': '1', 'eval_runtime': '27.94', 'eval_samples_per_second': '58.31', 'eval_steps_per_second': '29.18', 'epoch': '3'}
{'train_runtime': '3015', 'train_samples_per_second': '15.3', 'train_steps_per_second': '0.957', 'train_loss': '0.003084', 'epoch': '3'}
Restauração estrita aprovada: {'checkpoint': 'C:\\workspace\\python\\proj

       loss  grad_norm  ...  total_flos  train_loss
0  0.007548   0.013485  ...         NaN         NaN
1       NaN        NaN  ...         NaN         NaN
2  0.000662   0.005291  ...         NaN         NaN
3       NaN        NaN  ...         NaN         NaN
4  0.001042   0.001571  ...         NaN         NaN
5       NaN        NaN  ...         NaN         NaN
7       NaN        NaN  ...         NaN         NaN

[7 rows x 15 columns]

<Figure size 1200x400 with 2 Axes>

100%|#########9| 814/815 [00:27<00:00, 28.17it/s]
                                                  
 20%|##        | 577/2885 [17:25<57:13,  1.49s/it]
                                                 
100%|#########9| 812/815 [00:27<00:00, 28.86it/s]
                                                   
 40%|####      | 1154/2885 [33:52<41:55,  1.45s/it]
                                                 
100%|#########9| 812/815 [00:27<00:00, 28.80it/s]
                                                   
 60%|######    | 1731/2885 [50:13<28:44,  1.49s/it]
                                                 
Writing model shards: 100%|##########| 1/1 [00:00<00:00,  4.52it/s]


## Avaliação final e análise de erros

Depois de encerrar as escolhas, avaliamos os dois melhores checkpoints uma única vez no `Test` oficial, na ordem de `dados_teste`. Nenhum hiperparâmetro será ajustado com esses resultados. Relatamos accuracy, top-3, precision, recall e F1 macro/weighted. Os relatórios por classe, as matrizes normalizadas pela classe real e os erros de alta confiança ajudam a entender o que as médias escondem.

In [24]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, top_k_accuracy_score,
)

def calcular_metricas(rotulos, logits):
    previsoes = np.asarray(logits).argmax(axis=1)
    probabilidades = torch.softmax(torch.as_tensor(logits), dim=1).numpy()
    precisao_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        rotulos, previsoes, average="macro", zero_division=0
    )
    precisao_ponderada, recall_ponderado, f1_ponderado, _ = (
        precision_recall_fscore_support(
            rotulos, previsoes, average="weighted", zero_division=0
        )
    )
    return {
        "accuracy": accuracy_score(rotulos, previsoes),
        "top3_accuracy": top_k_accuracy_score(
            rotulos, probabilidades, k=3, labels=np.arange(QUANTIDADE_CLASSES)
        ),
        "precision_macro": precisao_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precisao_ponderada,
        "recall_weighted": recall_ponderado,
        "f1_weighted": f1_ponderado,
    }


saida_previsoes = treinador_fine_tuning.predict(
    conjunto_teste_pre_treinado, metric_key_prefix="test"
)
logits_vit_pre_treinado = np.asarray(saida_previsoes.predictions)
rotulos_teste = np.asarray(saida_previsoes.label_ids)
previsoes_vit_pre_treinado = logits_vit_pre_treinado.argmax(axis=1)
metricas_vit_pre_treinado = calcular_metricas(rotulos_teste, logits_vit_pre_treinado)


100%|##########| 1817/1817 [01:12<00:00, 25.06it/s]


### Inferência do modelo didático

Avaliamos o modelo no Test sem atualizar os pesos e mantemos a ordem das imagens.

In [25]:
@torch.inference_mode()
def prever_com_pytorch(modelo, carregador):
    modelo.eval()
    rotulos, logits = [], []
    for imagens, rotulos_lote in carregador:
        imagens = imagens.to(DISPOSITIVO, non_blocking=True)
        logits_lote = modelo(imagens)
        assert imagens.is_cuda and logits_lote.is_cuda
        logits.append(logits_lote.cpu())
        rotulos.append(rotulos_lote)
    return torch.cat(rotulos).numpy(), torch.cat(logits).numpy()


rotulos_vit_didatico, logits_vit_didatico = prever_com_pytorch(vit_didatico, carregador_teste)
assert np.array_equal(rotulos_vit_didatico, rotulos_teste)
previsoes_vit_didatico = logits_vit_didatico.argmax(axis=1)
metricas_vit_didatico = calcular_metricas(rotulos_vit_didatico, logits_vit_didatico)

comparacao = pd.DataFrame.from_dict(
    {"ViT do zero": metricas_vit_didatico, "ViT pré-treinado": metricas_vit_pre_treinado},
    orient="index",
)
display(comparacao)

relatorio_vit_didatico = pd.DataFrame(classification_report(
    rotulos_teste, previsoes_vit_didatico, labels=np.arange(QUANTIDADE_CLASSES),
    target_names=NOMES_CLASSES, zero_division=0, output_dict=True,
)).T
relatorio_vit_pre_treinado = pd.DataFrame(classification_report(
    rotulos_teste, previsoes_vit_pre_treinado, labels=np.arange(QUANTIDADE_CLASSES),
    target_names=NOMES_CLASSES, zero_division=0, output_dict=True,
)).T
comparacao_por_classe = pd.concat(
    {
        "ViT do zero": relatorio_vit_didatico.loc[NOMES_CLASSES, ["precision", "recall", "f1-score"]],
        "ViT pré-treinado": relatorio_vit_pre_treinado.loc[NOMES_CLASSES, ["precision", "recall", "f1-score"]],
    },
    axis=1,
)
display(comparacao_por_classe)


                  accuracy  top3_accuracy  ...  recall_weighted  f1_weighted
ViT do zero       0.996147            1.0  ...         0.996147     0.996162
ViT pré-treinado  0.990919            1.0  ...         0.990919     0.990707

[2 rows x 8 columns]

                      ViT do zero            ... ViT pré-treinado          
                        precision    recall  ...           recall  f1-score
Apple Braeburn 1         1.000000  1.000000  ...          1.00000  1.000000
Apple Golden 1           1.000000  1.000000  ...          1.00000  1.000000
Apple Granny Smith 1     1.000000  1.000000  ...          1.00000  1.000000
Apple Red Delicious 1    1.000000  1.000000  ...          1.00000  1.000000
Banana 1                 1.000000  1.000000  ...          1.00000  1.000000
Banana Lady Finger 1     1.000000  1.000000  ...          1.00000  1.000000
Banana Red 1             1.000000  1.000000  ...          1.00000  1.000000
Pepper Green 1           1.000000  1.000000  ...          1.00000  1.000000
Pepper Orange 1          1.000000  1.000000  ...          1.00000  1.000000
Pepper Red 1             1.000000  0.986486  ...          1.00000  1.000000
Pepper Yellow 1          1.000000  1.000000  ...          1.00000  1.000000
Tomato 1    

### Matrizes de confusão

Comparamos o recall de cada classe no mesmo Test. Os resultados continuam restritos a imagens com fundo regular.

In [26]:
matriz_vit_didatico = confusion_matrix(
    rotulos_teste, previsoes_vit_didatico, labels=np.arange(QUANTIDADE_CLASSES), normalize="true"
)
matriz_vit_pre_treinado = confusion_matrix(
    rotulos_teste, previsoes_vit_pre_treinado, labels=np.arange(QUANTIDADE_CLASSES), normalize="true"
)
figura, eixos = plt.subplots(1, 2, figsize=(26, 11))
for eixo, matriz, titulo in [
    (eixos[0], matriz_vit_didatico, "ViT do zero"),
    (eixos[1], matriz_vit_pre_treinado, "ViT pré-treinado"),
]:
    sns.heatmap(
        matriz, ax=eixo, cmap="Blues", vmin=0, vmax=1,
        xticklabels=NOMES_CLASSES, yticklabels=NOMES_CLASSES, cbar=False,
    )
    eixo.set(title=titulo, xlabel="Previsto", ylabel="Real")
    eixo.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()


<Figure size 2600x1100 with 2 Axes>

### Inspecionar erros

Exibimos até seis erros, começando pelos de maior probabilidade prevista. A galeria permite examinar as imagens, mas não demonstra a causa dos erros.

In [27]:
def mostrar_erros(logits, previsoes, titulo, quantidade=6):
    probabilidades = torch.softmax(torch.as_tensor(logits), dim=1).numpy()
    confianca = probabilidades[np.arange(len(previsoes)), previsoes]
    indices_erros = np.flatnonzero(previsoes != rotulos_teste)
    indices_ordenados = indices_erros[np.argsort(confianca[indices_erros])[::-1]][:quantidade]
    if len(indices_ordenados) == 0:
        print(f"{titulo}: nenhum erro no Test oficial.")
        return np.array([], dtype=int)
    figura, eixos = plt.subplots(2, 3, figsize=(13, 8))
    for eixo, indice in zip(eixos.flat, indices_ordenados):
        with Image.open(dados_teste.iloc[indice].path) as imagem:
            eixo.imshow(imagem.convert("RGB"))
        eixo.set_title(
            f"Real: {INDICE_PARA_CLASSE[int(rotulos_teste[indice])]}\n"
            f"Previsto: {INDICE_PARA_CLASSE[int(previsoes[indice])]} "
            f"({confianca[indice]:.3f})", fontsize=9,
        )
        eixo.axis("off")
    for eixo in eixos.flat[len(indices_ordenados):]:
        eixo.axis("off")
    figura.suptitle(f"Erros de maior confiança — {titulo}")
    plt.tight_layout()
    plt.show()
    return indices_ordenados


indices_erros_vit_didatico = mostrar_erros(
    logits_vit_didatico, previsoes_vit_didatico, "ViT do zero"
)
indices_erros_vit_pre_treinado = mostrar_erros(
    logits_vit_pre_treinado, previsoes_vit_pre_treinado, "ViT pré-treinado"
)


<Figure size 1300x800 with 6 Axes>

<Figure size 1300x800 with 6 Axes>

## Attention maps de um head individual

Inspecionamos o head 0 da última camada dos dois modelos na mesma imagem de teste. Mostramos a imagem, o heatmap dos 14 × 14 patches e a sobreposição. Attention weights indicam relações aprendidas entre tokens; não são máscaras de segmentação nem provam que uma região causou a previsão.

In [28]:
@torch.inference_mode()
def extrair_atencao_individual(modelo, imagem, pre_treinado):
    lote = imagem.unsqueeze(0).to(DISPOSITIVO)
    if pre_treinado:
        saida = modelo(pixel_values=lote, output_attentions=True)
        logits, atencoes = saida.logits, saida.attentions
    else:
        logits, atencoes = modelo(lote, retornar_atencoes=True)
    pesos = atencoes[-1][0, 0, 0, 1:].reshape(14, 14).float().cpu()
    return int(logits.argmax(1).item()), pesos.numpy()

vit_pre_treinado.gradient_checkpointing_disable()
vit_pre_treinado.eval()
INDICE_EXEMPLO_ATENCAO = 0
imagem = conjunto_teste_pre_treinado[INDICE_EXEMPLO_ATENCAO]["pixel_values"]
imagem_visivel = (imagem * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
figura, eixos = plt.subplots(2, 3, figsize=(12, 8))
registros_atencao = []
for linha, (nome, modelo, pre_treinado) in enumerate([
    ("ViT do zero", vit_didatico, False), ("ViT pré-treinado", vit_pre_treinado, True),
]):
    previsao, pesos = extrair_atencao_individual(modelo, imagem, pre_treinado)
    mapa = F.interpolate(torch.tensor(pesos)[None, None], size=(224, 224), mode="bilinear", align_corners=False)[0, 0].numpy()
    eixos[linha, 0].imshow(imagem_visivel)
    eixos[linha, 0].set_title(f"{nome}: {INDICE_PARA_CLASSE[previsao]}")
    eixos[linha, 1].imshow(pesos, cmap="magma")
    eixos[linha, 1].set_title("Head 0: CLS → patches")
    eixos[linha, 2].imshow(imagem_visivel)
    eixos[linha, 2].imshow(mapa, cmap="magma", alpha=0.45)
    eixos[linha, 2].set_title("Sobreposição na imagem")
    for eixo in eixos[linha]:
        eixo.axis("off")
    np.save(CAMINHO_RESULTADOS / f"attention_{linha}.npy", pesos)
    registros_atencao.append({"modelo": nome, "classe_real": INDICE_PARA_CLASSE[int(rotulos_teste[INDICE_EXEMPLO_ATENCAO])], "previsao": INDICE_PARA_CLASSE[previsao]})
plt.tight_layout()
caminho_figura_atencao = CAMINHO_RESULTADOS / "attention_examples.png"
figura.savefig(caminho_figura_atencao, dpi=120)
plt.show()
display(pd.DataFrame(registros_atencao))


<Figure size 1200x800 with 6 Axes>

             modelo       classe_real          previsao
0       ViT do zero  Apple Braeburn 1  Apple Braeburn 1
1  ViT pré-treinado  Apple Braeburn 1  Apple Braeburn 1

### O que aparece nesta imagem

A imagem 0 do Test mostra uma Apple Braeburn 1, reconhecida pelos dois modelos. No ViT do zero, o head 0 da última camada atribui mais peso ao fundo próximo do contorno do que ao interior da maçã. No pré-treinado, os pesos se concentram em patches dos cantos do fundo e em um ponto à direita da região central da fruta.

Neste exemplo, o head não pode ser interpretado como uma segmentação ou como atenção exclusiva à maçã. Cada heatmap tem sua própria escala. Comparamos a localização dos pesos, sem supor que o head 0 cumpra a mesma função nos dois modelos. Acertar a classe também não torna o mapa uma explicação causal.

A figura e esta inspeção pertencem à execução v3, com os checkpoints selecionados e conferidos. A restauração correta mantém, neste exemplo, o predomínio de pesos no fundo; a interpretação continua limitada a esta head e imagem.

## Resultados da execução v3 — 08/10/2026

| Modelo | Accuracy | F1 macro |
|---|---:|---:|
| ViT do zero | 99.6147% | 0.9968 |
| ViT pré-treinado | 99.0919% | 0.9908 |

O ViT do zero completou 8 épocas; a época 5 foi selecionada pelo F1 macro de validação (0.9951). O ViT pré-treinado teve a cabeça ajustada por duas épocas e o backbone completou três épocas de fine-tuning, com early stopping e seleção do checkpoint da primeira época pelo F1 macro de validação (1,0000). Cada restauração carregou os 200 tensores, sem chaves ausentes, inesperadas ou incompatíveis, e a validação recalculada coincidiu com a métrica registrada do estado selecionado.

Os dois modelos foram avaliados no mesmo Test oficial depois de encerrar as escolhas. As métricas desta tabela vêm da execução v3; os resultados v2 foram preservados em outra pasta. O ViT do zero obteve o maior F1 macro no Test neste recorte (0,9968 contra 0,9908). Entre os dois modelos avaliados, esse resultado e seu menor porte favorecem o modelo didático neste estudo. O F1 perfeito do pré-treinado na validação não garantiu maior desempenho no Test. Mantivemos a seleção pela validação, sem trocar checkpoints ou hiperparâmetros após observar o Test. Não demonstramos superioridade sobre CNNs nem generalização a cenas externas.

### Objetivos do pré-treinamento de BERT e ViT

| Modelo | Objetivo e representação |
|---|---|
| BERT original | Prediz tokens mascarados a partir do contexto bidirecional, maximizando a probabilidade dos tokens corretos; também usa next sentence prediction. Aprende representações textuais úteis para adaptação posterior. |
| ViT original | Classifica imagens rotuladas em grande escala, minimizando cross-entropy e aumentando a probabilidade da classe correta. Aprende representações visuais para transfer learning. |

Os dois usam Transformer. BERT processa subpalavras com contexto linguístico; ViT processa patches com informação espacial, para um objetivo de treinamento diferente.

### Contribuições de DeiT e Swin

DeiT usa regularização e augmentation para tornar o treinamento mais eficiente. Na variante com destilação, um token aprende com um modelo professor. Essas escolhas reduzem a dependência de grandes conjuntos externos de pré-treinamento.

Swin alterna janelas locais e deslocadas para conectar regiões vizinhas. Patch merging organiza as representações em escalas, com menor custo que a atenção global e uso em tarefas como detecção e segmentação.

### Escolha entre ViT e CNN neste domínio

CNNs usam filtros locais compartilhados, uma característica útil quando há poucos dados e orçamento limitado. ViTs podem aproveitar pré-treinamento amplo e relações entre regiões distantes. A escolha depende dos dados, da arquitetura e dos recursos disponíveis.

No Fruits-360, os objetos centralizados e o fundo uniforme tornam as cenas regulares. A comparação ajuda a escolher entre os dois ViTs avaliados, mas não demonstra superioridade sobre CNNs. Para testar a generalização, ainda precisamos de imagens com outros fundos e sessões de captura.

## Limitações e próximos passos

As imagens originais têm 100×100 pixels e fundo predominantemente branco. Redimensioná-las para 224×224 não cria detalhes. Quadros próximos podem compartilhar condições de captura, e as 20 classes selecionadas não cobrem toda a diversidade do Fruits-360 ou do varejo. O desempenho neste teste controlado precisa ser confirmado em prateleiras reais.

As confusões observadas devem orientar os próximos testes: coleta com iluminação, oclusão e fundos variados; divisão por sessão de captura; calibração de confiança; aumentos controlados; e comparação com um checkpoint menor para reduzir o custo. CutMix ficou desativado nesta execução.

## Auditoria final de reprodutibilidade

Conferimos os contratos do experimento, a separação dos conjuntos e as métricas calculadas. Também registramos as versões do ambiente usado na execução.

In [29]:
import platform
import transformers
import torchvision

assert torch.cuda.is_available()
assert QUANTIDADE_CLASSES == 20
assert caminhos_treino.isdisjoint(caminhos_validacao)
assert caminhos_treino.isdisjoint(caminhos_teste)
assert caminhos_validacao.isdisjoint(caminhos_teste)
assert logits_vit_didatico.shape == logits_vit_pre_treinado.shape == (len(dados_teste), QUANTIDADE_CLASSES)
assert np.array_equal(rotulos_vit_didatico, rotulos_teste)
assert set(metricas_vit_didatico) == set(metricas_vit_pre_treinado)
assert CHECKPOINT_VIT_DIDATICO.is_file()
assert (CAMINHO_MODELO_FINAL / "config.json").is_file()
assert (CAMINHO_MODELO_FINAL / "preprocessor_config.json").is_file()
assert caminho_figura_atencao.is_file()

registro_ambiente = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
}
print(registro_ambiente)
print("Auditoria aprovada: splits disjuntos, artefatos presentes e resultados finais consistentes.")


{'python': '3.12.6', 'torch': '2.9.0+cu130', 'torchvision': '0.24.0+cu130', 'transformers': '5.17.0', 'cuda': '13.0', 'gpu': 'NVIDIA GeForce RTX 3050 6GB Laptop GPU'}
Auditoria aprovada: splits disjuntos, artefatos presentes e resultados finais consistentes.


### Registrar os resultados desta versão

Salvamos as métricas e o ambiente para conferir a execução. Os arquivos auxiliares ficam apenas na máquina local.

A exportação final grava também `restauracao.json`, com os checkpoints escolhidos, a conferência dos 200 tensores e os F1 de validação selecionado e recalculado. Os valores pertencem à execução v3.

In [30]:
import json
comparacao.to_csv(CAMINHO_RESULTADOS / "comparacao.csv")
comparacao_por_classe.to_csv(CAMINHO_RESULTADOS / "metricas_por_classe.csv")
historico_vit_didatico.to_csv(CAMINHO_RESULTADOS / "historico_vit_didatico.csv", index=False)
(CAMINHO_RESULTADOS / "ambiente.json").write_text(json.dumps(registro_ambiente, indent=2), encoding="utf-8")

(CAMINHO_RESULTADOS / "restauracao.json").write_text(json.dumps({"cabeca": treinador_cabeca.auditoria_restauracao, "fine_tuning": treinador_fine_tuning.auditoria_restauracao}, indent=2), encoding="utf-8");
print("Exportação final conferida: métricas, históricos, ambiente e auditorias de restauração v3.")


Exportação final conferida: métricas, históricos, ambiente e auditorias de restauração v3.
